# 🔮 Guanacaste Tourism Forecaster — Fase 4: Cascade Forecasting
### Proyección de Llegadas y Ocupación 2026-2027 con Prophet
---
**Estrategia:** Cascade de 2 etapas (Entropía controlada)
- **Modelo A:** Proyectar llegadas aéreas (SJO + LIR) con regresores macro
- **Modelo B:** Proyectar ocupación Guanacaste usando llegadas proyectadas como regresor


In [1]:
import pandas as pd
import numpy as np
import warnings
import os
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.io as pio

# CONFIGURACION VS CODE RENDERER
# Si no ves los graficos, cambia 'iframe' por 'notebook' o 'vscode'
pio.renderers.default = 'iframe'

from prophet import Prophet

TEMPLATE = 'plotly_dark'
COLORS = {
    'total':    '#f9c74f',
    'ocupacion':'#ff6b6b',
    'forecast': '#a855f7',
    'ci':       'rgba(168,85,247,0.15)',
}

print('✅ Entorno listo. Directorio actual:', os.getcwd())


✅ Entorno listo. Directorio actual: C:\Users\franc\Downloads\guanacaste-tourism-forecaster\guanacaste-tourism-forecaster


---
## 1. Carga de Datos


In [2]:
# Intentar cargar datos con path relativo flexible
data_path = '../data/merged/master_tourism_data.csv'
if not os.path.exists(data_path):
    data_path = 'data/merged/master_tourism_data.csv'
    
df = pd.read_csv(data_path)
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values('DATE').reset_index(drop=True)
df['Arrivals_total'] = df['Arrivals_sjo'] + df['Arrivals_lir']

# Normalizacion
macro_cols = ['tasa_desempleo_usa', 'inflacion_usa_cpi', 'precio_petroleo_wti']
for col in macro_cols:
    mu, sigma = df[col].mean(), df[col].std()
    df[f'{col}_z'] = (df[col] - mu) / sigma
z_cols = [c + '_z' for c in macro_cols]
df[z_cols] = df[z_cols].ffill().bfill()

print(f'✅ Datos cargados: {len(df)} meses. Rango: {df.DATE.min().date()} a {df.DATE.max().date()}')


✅ Datos cargados: 206 meses. Rango: 2009-01-31 a 2026-02-28


In [3]:
def make_shock(df, start, end):
    return ((df['DATE'] >= pd.to_datetime(start)) & (df['DATE'] <= pd.to_datetime(end))).astype(float)

df['shock_covid'] = make_shock(df, '2020-03-01', '2021-05-31')
df['shock_huelga_2018'] = make_shock(df, '2018-09-01', '2018-12-31')
df['shock_inseguridad'] = make_shock(df, '2024-10-01', '2025-12-31')

SHOCK_COLS = ['shock_covid', 'shock_huelga_2018', 'shock_inseguridad']
print('✅ Shocks historicos definidos.')


✅ Shocks historicos definidos.


---
## 2. ETAPA 1: Llegadas Aéreas Totales


In [4]:
print('⏳ Entrenando Modelo A (Llegadas)... esto puede tardar unos segundos...')
df_A = df[['DATE', 'Arrivals_total'] + z_cols + SHOCK_COLS].rename(columns={'DATE':'ds', 'Arrivals_total':'y'})
m_A = Prophet(changepoint_prior_scale=0.1, yearly_seasonality=True)
for c in z_cols + SHOCK_COLS: m_A.add_regressor(c)
m_A.fit(df_A)

future_A = m_A.make_future_dataframe(periods=22, freq='ME')
for c in z_cols + SHOCK_COLS:
    mapping = df_A.set_index('ds')[c]
    future_A[c] = future_A['ds'].map(mapping)
    if c in z_cols: future_A[c] = future_A[c].fillna(df_A[c].iloc[-1])
    else: future_A[c] = future_A[c].fillna(0.0)

# Aplicar escenario de recuperacion 2026
mask_2026 = (future_A['ds'] >= '2026-01-01') & (future_A['ds'] <= '2026-12-31')
future_A.loc[mask_2026, 'shock_inseguridad'] = 0.4
future_A.loc[future_A['ds'] >= '2027-01-01', 'shock_inseguridad'] = 0.0

forecast_A = m_A.predict(future_A)
print('✅ Modelo A completado.')


⏳ Entrenando Modelo A (Llegadas)... esto puede tardar unos segundos...


13:08:59 - cmdstanpy - INFO - Chain [1] start processing


13:09:02 - cmdstanpy - INFO - Chain [1] done processing


✅ Modelo A completado.


In [5]:
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=df_A['ds'], y=df_A['y'], name='Histórico', mode='lines', line=dict(color=COLORS['total'])))
fig1.add_trace(go.Scatter(x=forecast_A['ds'], y=forecast_A['yhat'], name='Proyección', mode='lines', line=dict(color=COLORS['forecast'], dash='dash')))
fig1.update_layout(title='Pronóstico de Llegadas Internacionales (SJO+LIR)', template=TEMPLATE, hovermode='x unified')
fig1.show() # Usar .show() directo es mas compatible


---
## 3. ETAPA 2: Ocupación Hotelera Guanacaste (Cascade)


In [6]:
print('⏳ Entrenando Modelo B (Ocupación) usando la cascada...')
df_B = df[df.Guanacaste_Occupancy_Pct.notna()][['DATE','Guanacaste_Occupancy_Pct','Arrivals_total','shock_inseguridad']].rename(columns={'DATE':'ds','Guanacaste_Occupancy_Pct':'y'})
arr_mu, arr_sigma = df_B['Arrivals_total'].mean(), df_B['Arrivals_total'].std()
df_B['arr_z'] = (df_B['Arrivals_total'] - arr_mu) / arr_sigma

m_B = Prophet(changepoint_prior_scale=0.08, yearly_seasonality=True)
m_B.add_regressor('arr_z')
m_B.add_regressor('shock_inseguridad')
m_B.fit(df_B)

future_B = m_B.make_future_dataframe(periods=22, freq='ME')
arrivals_map = forecast_A.set_index('ds')['yhat']
future_B['arr_z'] = (future_B['ds'].map(arrivals_map).ffill() - arr_mu) / arr_sigma
shock_map = future_A.set_index('ds')['shock_inseguridad']
future_B['shock_inseguridad'] = future_B['ds'].map(shock_map).fillna(0.0)

forecast_B = m_B.predict(future_B)
forecast_B['yhat'] = forecast_B['yhat'].clip(0, 100)
print('✅ Modelo B completado.')


⏳ Entrenando Modelo B (Ocupación) usando la cascada...


13:09:04 - cmdstanpy - INFO - Chain [1] start processing


13:09:05 - cmdstanpy - INFO - Chain [1] done processing


✅ Modelo B completado.


In [7]:
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df_B['ds'], y=df_B['y'], name='Real', mode='lines+markers', line=dict(color=COLORS['ocupacion'])))
fig2.add_trace(go.Scatter(x=forecast_B['ds'], y=forecast_B['yhat'], name='Proyectado', mode='lines', line=dict(color=COLORS['forecast'])))
fig2.update_layout(title='Pronóstico de Ocupación Hotelera Guanacaste (%)', template=TEMPLATE, yaxis=dict(range=[0,105]))
fig2.show()


---
## 4. Tabla de Resultados (Proyección 2026)


In [8]:
res = forecast_B[forecast_B['ds'] > '2025-12-31'][['ds','yhat']].head(12)
res.columns = ['Mes','Ocupacion_Pct']
res['Mes'] = res['Mes'].dt.strftime('%B %Y')
print(res.to_string(index=False))


           Mes  Ocupacion_Pct
  January 2026      74.481652
 February 2026      78.085976
    March 2026      71.980140
    April 2026      67.403962
      May 2026      61.251692
     June 2026      58.984847
     July 2026      62.644779
   August 2026      55.173372
September 2026      43.559749
  October 2026      42.153882
